# Open Catalyst Graph Regression with GNNVisualizer

This notebook builds graph-level GCN, GAT, GraphSAGE, and GIN models for Open Catalyst style structure-to-energy regression, then renders the four models with `GNNVisualizer`.

Open Catalyst datasets such as OC20 are large. The notebook will read a local FairChem/ASE database if `OCP_DATA_PATH` is set. Without that path it falls back to a tiny built-in catalyst preview set so the modeling and visualization cells still run end to end.

Source docs: [Open Catalyst datasets](https://fair-chem.github.io/catalysts/datasets/summary.html) and [OC20 data notes](https://fair-chem.github.io/catalysts/datasets/oc20.html).

Optional real-data setup:

```bash
python3 -m pip install torch torch-geometric fairchem-core ase
export OCP_DATA_PATH=/path/to/ocp_or_fairchem_dataset
```

The fallback preview graphs use atomic numbers and Cartesian positions as node features, and radius edges computed from atom positions. For production OC20 or OC22 work, use the official FairChem data loaders and task-specific splits.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, GCNConv, GINConv, SAGEConv, global_mean_pool

from gnn_exp import GNNVisualizer

from torch_geometric.data import Data


In [ ]:
SEED = 7
torch.manual_seed(SEED)

OCP_DATA_PATH = os.environ.get("OCP_DATA_PATH")
MAX_GRAPHS = int(os.environ.get("OCP_MAX_GRAPHS", "64"))
BATCH_SIZE = 8
EPOCHS = 12
HIDDEN_CHANNELS = 16
RADIUS_CUTOFF = float(os.environ.get("OCP_RADIUS_CUTOFF", "2.8"))


def radius_edge_index(pos, cutoff=RADIUS_CUTOFF):
    distances = torch.cdist(pos, pos)
    mask = (distances <= cutoff) & (distances > 0)
    edge_index = mask.nonzero(as_tuple=False).t().contiguous()
    if edge_index.numel() > 0:
        return edge_index
    nearest = distances + torch.eye(pos.size(0), device=pos.device) * 1e9
    targets = nearest.argmin(dim=1)
    edges = torch.stack([torch.arange(pos.size(0), device=pos.device), targets], dim=0)
    return torch.cat([edges, edges.flip(0)], dim=1).contiguous()


def graph_from_atomic_structure(atomic_numbers, pos, energy):
    atomic_numbers = torch.as_tensor(atomic_numbers, dtype=torch.long)
    pos = torch.as_tensor(pos, dtype=torch.float)
    centered_pos = pos - pos.mean(dim=0, keepdim=True)
    position_scale = centered_pos.abs().max().clamp_min(1.0)
    x = torch.cat([atomic_numbers.float().view(-1, 1) / 100.0, centered_pos / position_scale], dim=1)
    return Data(
        x=x,
        edge_index=radius_edge_index(pos),
        y=torch.tensor([float(energy)], dtype=torch.float),
        atomic_numbers=atomic_numbers,
        pos=pos,
    )


def make_preview_dataset():
    base_z = torch.tensor([78, 78, 78, 78, 29, 29, 8, 1, 1], dtype=torch.long)
    base_pos = torch.tensor([
        [-1.35, -1.35, 0.00],
        [ 1.35, -1.35, 0.00],
        [-1.35,  1.35, 0.00],
        [ 1.35,  1.35, 0.00],
        [ 0.00, -2.70, 0.10],
        [ 0.00,  2.70, 0.10],
        [ 0.10,  0.00, 1.75],
        [ 0.85,  0.20, 2.25],
        [-0.70, -0.25, 2.35],
    ], dtype=torch.float)
    graphs = []
    for index in range(12):
        shift = (index - 5.5) * 0.035
        pos = base_pos.clone()
        pos[6:, 0] += shift
        pos[6:, 2] += 0.04 * torch.sin(torch.tensor(float(index)))
        energy = -1.25 + 0.08 * index + 0.15 * float(pos[6:, 2].mean())
        graphs.append(graph_from_atomic_structure(base_z, pos, energy))
    return graphs


def get_record_value(record, names):
    for name in names:
        if isinstance(record, dict) and name in record:
            return record[name]
        if hasattr(record, name):
            return getattr(record, name)
    return None


def record_to_graph(record):
    if hasattr(record, "get_atomic_numbers") and hasattr(record, "get_positions"):
        energy = record.get_potential_energy() if hasattr(record, "get_potential_energy") else 0.0
        return graph_from_atomic_structure(record.get_atomic_numbers(), record.get_positions(), energy)
    z = get_record_value(record, ["atomic_numbers", "z"])
    pos = get_record_value(record, ["pos", "positions"])
    energy = get_record_value(record, ["y", "energy", "relaxed_energy", "y_relaxed"])
    if z is None or pos is None:
        raise ValueError("Could not find atomic numbers and positions in the OCP record")
    if isinstance(energy, torch.Tensor):
        energy = energy.detach().view(-1)[0].item()
    if energy is None:
        energy = 0.0
    return graph_from_atomic_structure(z, pos, energy)


def load_ocp_graphs():
    if not OCP_DATA_PATH:
        return make_preview_dataset(), "built-in preview structures"
    try:
        from fairchem.core.datasets import AseDBDataset
    except ImportError as exc:
        raise ImportError("Install fairchem-core or unset OCP_DATA_PATH to use the built-in preview graphs") from exc
    dataset = AseDBDataset(config=dict(
        src=OCP_DATA_PATH,
        a2g_args=dict(r_energy=True, r_forces=False),
        keep_in_memory=False,
    ))
    count = min(MAX_GRAPHS, len(dataset))
    graphs = [record_to_graph(dataset[index]) for index in range(count)]
    return graphs, f"FairChem dataset at {OCP_DATA_PATH}"


graphs, dataset_source = load_ocp_graphs()
train_size = max(1, int(0.8 * len(graphs)))
train_graphs = graphs[:train_size]
valid_graphs = graphs[train_size:] or graphs[:1]
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=BATCH_SIZE, shuffle=False)
visual_data = graphs[0]
QUERY_PAIR = [0, min(6, visual_data.num_nodes - 1)]
NUM_FEATURES = visual_data.num_features
OUT_CHANNELS = 1

target_values = torch.stack([graph.y.view(()) for graph in train_graphs])
target_mean = target_values.mean()
target_std = target_values.std().clamp_min(1e-6)

display(Markdown(
    f"Using **{dataset_source}** with **{len(graphs)} graph(s)**. "
    f"The visualized structure has **{visual_data.num_nodes} atoms**, "
    f"**{visual_data.edge_index.size(1)} radius edges**, and **{NUM_FEATURES} node features**."
))


In [ ]:
class GCNGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


class GATGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        if hidden_channels % 2 != 0:
            raise ValueError("hidden_channels must be divisible by 2")
        heads = 2
        per_head_channels = hidden_channels // heads
        self.conv1 = GATConv(in_channels, per_head_channels, heads=heads, concat=True)
        self.act1 = nn.Tanh()
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1, concat=False)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


class GraphSAGEGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


class GINGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GINConv(nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, hidden_channels),
        ))
        self.act1 = nn.Tanh()
        self.conv2 = GINConv(nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, hidden_channels),
        ))
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


In [ ]:
def scaled_target(batch):
    return (batch.y.view(-1, 1).float() - target_mean) / target_std


def evaluate_model(model, loader):
    model.eval()
    absolute_errors = []
    with torch.no_grad():
        for batch in loader:
            scaled_pred = model(batch.x, batch.edge_index, batch.batch)
            pred = scaled_pred * target_std + target_mean
            absolute_errors.append((pred.view(-1) - batch.y.view(-1).float()).abs())
    return float(torch.cat(absolute_errors).mean())


def train_model(model, loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
    losses = []
    for _ in range(epochs):
        model.train()
        for batch in loader:
            optimizer.zero_grad()
            pred = model(batch.x, batch.edge_index, batch.batch)
            loss = F.mse_loss(pred, scaled_target(batch))
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
    return {"final_scaled_mse": losses[-1], "valid_mae": evaluate_model(model, valid_loader)}


model_builders = {
    "GCN": lambda: GCNGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
    "GAT": lambda: GATGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
    "GraphSAGE": lambda: GraphSAGEGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
    "GIN": lambda: GINGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
}

models = {}
metrics_by_model = {}
for name, build_model in model_builders.items():
    torch.manual_seed(SEED)
    model = build_model()
    metrics_by_model[name] = train_model(model, train_loader)
    models[name] = model.eval()

rows = ["| Model | Final scaled MSE | Validation MAE |", "|---|---:|---:|"]
for name, metrics in metrics_by_model.items():
    rows.append(f"| {name} | {metrics['final_scaled_mse']:.4f} | {metrics['valid_mae']:.4f} |")
display(Markdown("\n".join(rows)))


The next cell creates one `GNNVisualizer` per trained model. The query pair highlights a surface atom and an adsorbate atom when the preview graph is used.

In [ ]:
EXPECTED_LAYER_TYPES = {
    "GCN": "GCNConv",
    "GAT": "GATConv",
    "GraphSAGE": "SAGEConv",
    "GIN": "GINConv",
}


def make_visualizer(model, graph_data, query_pair):
    visualizer = GNNVisualizer(renderer="svg")
    visualizer.add_model(
        data=graph_data,
        model=model.eval(),
        subgraphSample=False,
        queries=[query_pair],
        mode="graph",
    )
    return visualizer


visualizers = {
    name: make_visualizer(model, visual_data, QUERY_PAIR)
    for name, model in models.items()
}

summary_rows = [
    "| Model | First layer | Aggregation | Graph pooling | Hidden width | Visualized nodes | Query |",
    "|---|---:|---:|---:|---:|---:|---:|",
]

for name, visualizer in visualizers.items():
    first_layer = visualizer.modelInfo["conv1"]
    assert first_layer["type"] == EXPECTED_LAYER_TYPES[name]
    assert len(visualizer.graphData["x"]) == visual_data.num_nodes
    assert "graphAggregation" in visualizer.intmData
    assert len(visualizer.intmData["act1"][0]) == HIDDEN_CHANNELS
    summary_rows.append(
        f"| {name} | `{first_layer['type']}` | `{first_layer.get('aggregation')}` | "
        f"`{visualizer.intmData['graphAggregation']['type']}` | "
        f"{len(visualizer.intmData['act1'][0])} | {len(visualizer.graphData['x'])} | "
        f"`{visualizer.queries}` |"
    )

display(Markdown("\n".join(summary_rows)))


## GCN

In [ ]:
display(visualizers["GCN"])

## GAT

In [ ]:
display(visualizers["GAT"])

## GraphSAGE

In [ ]:
display(visualizers["GraphSAGE"])

## GIN

In [ ]:
display(visualizers["GIN"])